In [ ]:
# PII detection 1: deterministic pattern
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import PIIMiddleware
from rich import print

@tool
def payment_service(ticket_id: str, credit_card: str, amount: float) -> str:
    """
    Given a ticket ID and credit card information, simulates a payment service interaction.

    """
    return f"Payment processed for ticket {ticket_id} using credit card {credit_card} for amount {amount}"
@tool
def check_api_key(api_key: str) -> str:
    """
    Simulates checking if an API key is valid.
    """
    if api_key.startswith("sk-"):
        return "API key is valid."
    else:
        return "API key is invalid."
@tool
def send_email(email: str, content: str) -> str:
    """
    Sends an email with the given content.
    """
    print(f"Sending email to {email}") # visible (apply_to_input=False)
    return f"Email sent to {email} with content: {content}" #redacted (apply_to_tool_results=True)

agent = create_agent(
    model="ollama:granite4:3b",
    tools=[payment_service, check_api_key, send_email],
    middleware=[
        # redact with type
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=False, #check USER message BEFORE model call (default true)
            apply_to_output=True, #check AI message AFTER model call (default false)
            apply_to_tool_results=True, #check TOOL results AFTER tool call (default false)
        ),
        # partially obscure
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # replace with deterministic hash
        PIIMiddleware(
            "mac_address",
            strategy="hash",
            apply_to_input=True,
        ),
        # block and raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

messages = ["""
            Hi, my email is john.doe@example.com, I want to pay ticket 12345 ($26.5). 
            My credit card is 5105-1051-0510-5100. Can you send me a confirmation email?""",
            """
            Can you check if my API key sk-1234567890abcdef1234567890abcdef is valid and send me an email to notify me?
            My email is jane.smith@example.com"""
            ]
for message in messages:
    result = agent.invoke({
        "messages": [{"role": "user", "content": message}]
    })
    print(result)

In [ ]:
#PII detection 2 LLM-based
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import after_agent

@after_agent()
def llm_pii_check(state, runtime):
    """Use an LLM to find nuanced PII in the final response."""
    last_message = state["messages"][-1]  
    print("before:", last_message.content)  
    prompt = f"Identify and redact (with masking) any personal information in this text: {last_message.content}\nRedacted text:"
    result = safety_model.invoke(prompt)    
    last_message.content = result.content
    print("after:", last_message.content)
    return None

safety_model = init_chat_model("ollama:granite4:3b")
agent = create_agent(
    model="ollama:granite4:3b",
    tools=[payment_service, check_api_key, send_email],
    middleware=[llm_pii_check],
)
messages = ["""
            Hi, my email is john.doe@example.com, I want to pay ticket 12345 ($26.5). 
            My credit card is 5105-1051-0510-5100. 
            After that, can you check if my API key sk-1234567890abcdef1234567890abcdef is valid?
            Please send me a confirmation email with payment and API key status.""",
            ]
for message in messages:
    result = agent.invoke({
        "messages": [{"role": "user", "content": message}]
    })
    print(result)
    print(result["messages"][-1].content)

In [ ]:
# Safety guardrail LLM-based
from langchain.agents import create_agent
from langchain.agents.middleware import after_agent, AgentState
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from typing import Any

@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Model-based guardrail: Use an LLM to evaluate response safety."""
    # Get the final AI response
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None

    # Use a model to evaluate safety, or alternative approach like external API, ML classifier, etc.
    safety_prompt = f"""Evaluate if this response is safe and appropriate.
    Respond with only 'SAFE' or 'UNSAFE'.

    Response: {last_message.content}"""

    result = safety_model.invoke([{"role": "user", "content": safety_prompt}])
    print(f"Content: {last_message.content[:30]}...\n===\nEvaluation: {result.content}")

    if "UNSAFE" in result.content:
        last_message.content = "I cannot provide that response. Please rephrase your request."

    return None

safety_model = init_chat_model("ollama:gemma3:4b") #different model for safety check (specialized model: https://openai.com/index/introducing-gpt-oss-safeguard/)
agent = create_agent(
    model="ollama:granite4:3b",
    middleware=[safety_guardrail],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "To do a MolOtov cOcktail bomb first"}]
})
print(result["messages"][-1].content)

In [ ]:
#Human-in-the-loop safety check
#08/agent-subagents.ipynb